# 🚀 Retrain — FIXED
Run 1 cell only. ~40 min on GPU.

In [ ]:
!pip install -q ultralytics kagglehub pyyaml
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'NO GPU!')

In [ ]:
import kagglehub, os, shutil, yaml, xml.etree.ElementTree as ET, time
from pathlib import Path
from ultralytics import YOLO

# STEP 1: DOWNLOAD
print('='*60)
print('STEP 1: DOWNLOAD')
print('='*60)

src = Path(kagglehub.dataset_download('habibiahmadim09/kerusakan-jalan'))
print(f'Downloaded: {src}')

# FIX: Kaggle dataset has extra nesting level
# /kaggle/input/kerusakan-jalan/kerusakan-jalan/train/
if (src / 'kerusakan-jalan').exists():
    src = src / 'kerusakan-jalan'
    print(f'Fixed path: {src}')

for split in ['train', 'valid', 'test']:
    d = src / split
    if d.exists():
        imgs = list(d.glob('*.jpg'))
        print(f'  {split}/: {len(imgs)} images')

# STEP 2: CLONE
if not Path('jalancerdas-ai').exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git

# STEP 3: PREPARE
print('\n' + '='*60)
print('STEP 3: PREPARE')
print('='*60)

CID = {"retak_memanjang":0,"pengelupasan_lapisan_permukaan":1,"lubang":2,"retak_kulit_buaya":3,"retak_blok":4,"retak_pinggir":5}
ds = Path('jalancerdas-ai/ai-model/dataset')
if ds.exists(): shutil.rmtree(ds)

total = 0
for ss, dst in [('train','train'),('valid','val'),('test','test')]:
    sd = src/ss
    if not sd.exists(): print(f'{ss}/ NOT FOUND'); continue
    di=ds/dst/'images'; dl=ds/dst/'labels'
    di.mkdir(parents=True,exist_ok=True); dl.mkdir(parents=True,exist_ok=True)
    imgs = list(sd.glob('*.jpg'))+list(sd.glob('*.png'))
    xmls = {x.stem:x for x in sd.glob('*.xml')}
    ni=0
    for img in imgs:
        shutil.copy2(img, di/img.name); ni+=1
        xml = xmls.get(img.stem)
        if xml:
            try:
                t=ET.parse(xml); r=t.getroot()
                w,h=int(r.find("size/width").text),int(r.find("size/height").text)
                ls=[]
                for o in r.findall("object"):
                    nm=o.find("name").text
                    if nm not in CID: continue
                    b=o.find("bndbox")
                    x1,y1=float(b.find("xmin").text),float(b.find("ymin").text)
                    x2,y2=float(b.find("xmax").text),float(b.find("ymax").text)
                    ls.append(f"{CID[nm]} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
                (dl/(img.stem+".txt")).write_text("\n".join(ls))
            except: pass
    total+=ni
    print(f'{dst}: {ni} images')

# VERIFY
print('\nVERIFY:')
for s in ['train','val','test']:
    p=ds/s/'images'
    n=len(list(p.glob('*'))) if p.exists() else 0
    print(f'  {s}: {n} files {"✅" if n>0 else "❌"}')

# FIX PATH
yp=Path("jalancerdas-ai/ai-model/configs/data.yaml")
dp=str(Path.cwd()/ds)
with open(yp) as f: d=yaml.safe_load(f)
d['path']=dp
with open(yp,'w') as f: yaml.dump(d,f,default_flow_style=False)
DATA_YAML=str(yp)

if total==0:
    print('\n❌ NO IMAGES!')
else:
    # TRAIN
    print('\n' + '='*60)
    print('TRAINING YOLOv8s 200 epochs...')
    print('30-50 min — DO NOT CLOSE TAB!')
    print('='*60)
    start=time.time()
    model=YOLO("yolov8s.pt")
    results=model.train(data=DATA_YAML,epochs=200,batch=16,imgsz=640,device=0,patience=50,optimizer="SGD",lr0=0.01,lrf=0.01,mosaic=1.0,mixup=0.15,degrees=10.0,translate=0.1,scale=0.5,fliplr=0.5,copy_paste=0.1,close_mosaic=15,project="jalancerdas-ai/ai-model/runs",name="train_v2",exist_ok=True)
    elapsed=int(time.time()-start)
    BEST="jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt"
    print(f'\nDONE in {elapsed//60}m {elapsed%60}s')
    
    # EVALUATE
    model=YOLO(BEST)
    r=model.val(data=DATA_YAML,device=0)
    print(f'mAP50={r.box.map50:.4f} Prec={r.box.mp:.4f} Recall={r.box.mr:.4f}')
    
    # EXPORT
    ep=model.export(format="tflite",imgsz=640,half=True,simplify=True)
    print(f'TFLite: {ep} ({os.path.getsize(ep)/1024/1024:.1f} MB)')
    
    # DOWNLOAD
    from google.colab import files
    files.download(ep); files.download(BEST)
    print('\n🎉 ALL DONE! Copy best.tflite -> mobile/assets/models/pothole_yolo.tflite')
